In [1]:
# MATSRL/train_synthesizer.py
# --- 0. Install necessary libraries (if running in a fresh environment like Colab) ---
!pip install transformers datasets rouge-score torch accelerate -q

# --- 1. Imports ---
import torch
from datasets import load_dataset, concatenate_datasets
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq, BartTokenizer # Need BartTokenizer here for preprocessing

# Import your custom modules
from agent_synthesizer import SynthesizerAgent
from trainer_a2c import A2CTrainer
# reward_utils is imported by SynthesizerAgent, so no need to import it here directly unless you use its functions here.

# --- 2. Configuration ---
MODEL_NAME = 'facebook/bart-base'
LEARNING_RATE = 3e-5
NUM_EPOCHS = 3
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 8 # Accumulate gradients over 8 batches to simulate BATCH_SIZE * 8
LOG_INTERVAL = 10 # Log every X accumulation steps (not per raw batch)
SAVE_MODEL_EVERY_EPOCH = True # Set to True to save model checkpoints after each epoch
SAVE_PATH = "./trained_synthesizer_agent.pth" # Path to save the final model

# --- 3. Device Setup ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training will run on: {DEVICE}")

# --- 4. Initialize the SynthesizerAgent ---
# The tokenizer is part of the agent now, but we need it initialized to pass to DataCollatorForSeq2Seq.
agent = SynthesizerAgent(
    model_name=MODEL_NAME,
    lr=LEARNING_RATE,
    device=DEVICE,
    max_output_len=120,
    max_input_len=512
)
print(f"Synthesizer Agent initialized with model: {MODEL_NAME}")

# --- 5. Load and Prepare Datasets ---
print("\n--- Loading and preparing datasets ---")

# List of datasets to use
dataset_configs = [
    ("cnn_dailymail", "3.0.0"),
    "xsum",
    "billsum",
    ("arxiv", "v1.1")
]

raw_datasets = {}
for name_config in dataset_configs:
    if isinstance(name_config, tuple):
        ds_name, ds_config = name_config
        print(f"Loading {ds_name} ({ds_config})...")
        raw_datasets[ds_name] = load_dataset(ds_name, ds_config, split="train")
    else:
        print(f"Loading {name_config}...")
        raw_datasets[name_config] = load_dataset(name_config, split="train")

# Preprocessing function for all datasets
def preprocess_function(examples):
    # Adjust column names based on dataset.
    inputs = []
    targets = []

    # Handle common column names for input
    if "article" in examples:
        inputs = examples["article"]
    elif "document" in examples:
        inputs = examples["document"]
    elif "text" in examples: # For some generic text datasets (less common for summarization)
        inputs = examples["text"]
    else:
        # Fallback or raise error if input column not found
        raise ValueError(f"No suitable input column found in examples: {examples.keys()}")

    # Handle common column names for target
    if "highlights" in examples:
        targets = examples["highlights"]
    elif "summary" in examples:
        targets = examples["summary"]
    elif "abstract" in examples: # For datasets like arXiv, 'abstract' is the target
        targets = examples["abstract"]
    else:
        raise ValueError(f"No suitable target column found in examples: {examples.keys()}")

    # Tokenize inputs and targets using the agent's tokenizer
    model_inputs = agent.tokenizer(
        inputs, max_length=agent.max_input_len, truncation=True, padding=False
    )

    labels = agent.tokenizer(
        text_target=targets, max_length=agent.max_output_len, truncation=True, padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_datasets = []
for name, dataset in raw_datasets.items():
    print(f"Tokenizing {name} dataset...")
    # Use map with batched=True for faster tokenization
    tokenized_ds = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names, # Remove original columns to save memory
        num_proc=2 # Adjust based on your CPU cores for faster processing
    )
    tokenized_datasets.append(tokenized_ds)

# Combine all tokenized datasets
if len(tokenized_datasets) > 1:
    combined_dataset = concatenate_datasets(tokenized_datasets)
    print(f"Combined dataset size: {len(combined_dataset)} examples")
else:
    combined_dataset = tokenized_datasets[0]
    print(f"Single dataset size: {len(combined_dataset)} examples")


# Create a DataLoader for batching
# The DataCollatorForSeq2Seq will pad inputs to the longest sequence in the batch.
data_collator = DataCollatorForSeq2Seq(agent.tokenizer, model=agent.model)

train_dataloader = DataLoader(
    combined_dataset,
    shuffle=True,
    batch_size=BATCH_SIZE,
    collate_fn=data_collator,
    num_workers=2 # Adjust for better data loading performance
)
print(f"DataLoader created with batch size {BATCH_SIZE}")

# --- 6. Initialize and Run the A2CTrainer ---
print("\n--- Starting A2C Training ---")

trainer = A2CTrainer(
    agent=agent,
    dataloader=train_dataloader,
    num_epochs=NUM_EPOCHS,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    log_interval=LOG_INTERVAL,
    device=DEVICE
)

trained_rewards = trainer.train()
print(f"Training completed. Average rewards per epoch: {trained_rewards}")

# --- 7. Save the final trained model ---
# It's good practice to save the model's state_dict, not the entire model object
# to avoid issues with different code versions or loading on different systems.
torch.save(agent.state_dict(), SAVE_PATH)
print(f"Final trained agent saved to {SAVE_PATH}")

# --- 8. Optional: Test the agent after training (inference mode) ---
print("\n--- Testing Trained Agent (Inference) ---")
agent.eval() # Set model to evaluation mode for inference
test_input_text = "The recent scientific discovery regarding quantum computing promises to revolutionize information processing and cryptography, leading to faster computations and more secure communications."
# Use beam search for inference, as it typically produces higher quality summaries than stochastic sampling
final_summary = agent.generate_summary(test_input_text, num_beams=4)
print(f"Input for Testing: {test_input_text}")
print(f"Generated Summary by Trained Agent: {final_summary}")

ModuleNotFoundError: No module named 'agent_synthesizer'